# Course 1 · Week 1 — Hands-on: Gradient Descent

> **Open in Colab badge will be added once the file is on GitHub.**

This notebook is the practical companion to **Week 1** of [course1.html](https://github.com/waytan8288/machine-learning-guide). You read about gradient descent there; now you're going to *implement* it.

By the end you will have:

1. Generated 10 synthetic training points (a noisy line)
2. Implemented the prediction function `f(x) = wx + b`
3. Implemented the squared-error cost function `J(w, b)`
4. Implemented the partial derivatives (gradients)
5. Implemented gradient descent that finds `w` and `b` automatically
6. Watched what happens when the learning rate is too small or too large

**How to use this notebook:** every code cell with a `# TODO` comment is something you fill in. The cells right below each implementation will sanity-check your code automatically — they'll fail loudly if your implementation is wrong, and print a ✓ if it's right. Look at the [solution notebook](../solutions/course1-week1-gradient-descent-solution.ipynb) only after you've genuinely tried.

Estimated time: **45–60 minutes.**


## Setup — generate the data

We're going to fit a line to 10 noisy points. We *know* the true line we sampled from is `y = 2x + 5` (because we made it up), so we can check at the end whether gradient descent recovered it.

In the real world you of course don't know the true line — that's the whole point of fitting one.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reproducible random data — DO NOT change these lines
np.random.seed(42)
m = 10  # number of training examples
X = np.linspace(0.5, 5.0, m)            # input feature (e.g. house size in 1000s of sq ft)
y_true = 2.0 * X + 5.0                  # the true line: slope 2, intercept 5
y = y_true + np.random.normal(0, 0.8, m)  # observed values with noise (e.g. price in $100k)

print(f"X = {np.round(X, 2)}")
print(f"y = {np.round(y, 2)}")


You should see 10 points roughly arranged along an upward slope, but not on a perfect line — that's the noise. Your job is to find the best straight line through this cloud.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X, y, label="Training data", color="#4338ca", s=80, zorder=3)
plt.xlabel("X")
plt.ylabel("y")
plt.title("Synthetic training data")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Quick recap

Linear regression has three ingredients:

1. **A model.** Here it's `f(x) = w*x + b`. Two parameters: `w` (slope) and `b` (intercept).
2. **A cost function.** Squared error: J(w, b) = (1/2m) * sum( (f(x_i) - y_i)^2 ). Bigger when the line misses points; zero when it goes through every point exactly.
3. **An optimizer.** Gradient descent: at each step, look at how cost changes if you nudge `w` or `b`, then take a small step in the direction that *reduces* cost. Repeat thousands of times.

The math is in [Week 1 of the course guide](#). We're implementing it from scratch here — no scikit-learn, no high-level libraries. Just numpy and the formulas you already know.

## Exercise 1 — the prediction function

Implement `predict(x, w, b)` so that it computes `f(x) = w*x + b`.

`x` will sometimes be a single number and sometimes a numpy array — your implementation should work for both. (Numpy makes this easy; the same expression `w * x + b` works for scalars and arrays.)


In [ ]:
def predict(x, w, b):
    """Compute the linear model: f(x) = w*x + b.

    Args:
        x: a single number, or a numpy array of inputs.
        w: scalar — the weight (slope).
        b: scalar — the bias (intercept).

    Returns:
        Same shape as x — the predicted values.
    """
    # TODO: replace this with the correct one-line expression
    return 0.0


# Quick sanity check — should print 7.0
print(f"predict(3.0, 2.0, 1.0) = {predict(3.0, 2.0, 1.0)}")
assert abs(predict(3.0, 2.0, 1.0) - 7.0) < 1e-9, "predict(3, 2, 1) should be 7.0"
print("✓ predict() looks correct")


## Exercise 2 — the cost function

Implement `cost(X, y, w, b)` so that it returns the **average squared error**, divided by 2:

$$J(w, b) = \frac{1}{2m} \sum_{i=1}^{m} \bigl(f_{w,b}(x_i) - y_i\bigr)^2$$

Use your `predict()` function. With numpy you can subtract two arrays (`yhat - y`), square element-wise (`** 2`), and average (`np.mean(...)`) — three lines or fewer.

If your implementation is right, the test cell will print two cost values close to **63.18** and **0.21**. Why those specific numbers? Because the data is fixed (we set the random seed) and `(w=0, b=0)` predicts zero everywhere — way off — while `(w=2, b=5)` happens to be very close to the true line.


In [ ]:
def cost(X, y, w, b):
    """Squared-error cost J(w, b) = (1/2m) * sum((f(x_i) - y_i)^2).

    Args:
        X: numpy array of inputs, shape (m,).
        y: numpy array of targets, shape (m,).
        w, b: scalars.

    Returns:
        A single float — the cost.
    """
    # TODO: implement using the predict() function you wrote above.
    # Hint: numpy lets you write   yhat - y   and get a vector of errors.
    return 0.0


# Sanity checks (these come from the reference solution):
J_zero = cost(X, y, 0.0, 0.0)
J_good = cost(X, y, 2.0, 5.0)
print(f"cost at (w=0, b=0) = {J_zero:.4f}   (expected ≈ 63.18)")
print(f"cost at (w=2, b=5) = {J_good:.4f}   (expected ≈ 0.21)")

assert abs(J_zero - 63.1830) < 0.01, "cost(0,0) should be ≈ 63.18"
assert abs(J_good - 0.2148) < 0.01,  "cost(2,5) should be ≈ 0.21"
print("✓ cost() looks correct")


## Exercise 3 — the gradients

Now we need to know which direction to nudge `w` and `b` to reduce the cost. The math gives us:

$$\frac{\partial J}{\partial w} = \frac{1}{m} \sum_{i=1}^{m} \bigl(f(x_i) - y_i\bigr) x_i \qquad \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} \bigl(f(x_i) - y_i\bigr)$$

In words: both gradients are an average of errors. The gradient w.r.t. `w` weights each error by its `x`-value (because changing `w` affects predictions in proportion to `x`). The gradient w.r.t. `b` is the unweighted average error.

Implement `gradients(X, y, w, b)` returning a tuple `(dw, db)`.


In [ ]:
def gradients(X, y, w, b):
    """Partial derivatives of J w.r.t. w and b.

    dJ/dw = (1/m) * sum( (f(x_i) - y_i) * x_i )
    dJ/db = (1/m) * sum(  f(x_i) - y_i        )

    Returns:
        (dw, db) — two scalars.
    """
    # TODO: compute the two partial derivatives.
    dw = 0.0
    db = 0.0
    return dw, db


dw0, db0 = gradients(X, y, 0.0, 0.0)
print(f"At (w=0, b=0): dw = {dw0:.4f}, db = {db0:.4f}   (expected ≈ -33.96, -10.86)")

assert abs(dw0 - (-33.9630)) < 0.01, "dw at (0,0) should be ≈ -33.96"
assert abs(db0 - (-10.8584)) < 0.01, "db at (0,0) should be ≈ -10.86"
print("✓ gradients() looks correct")


## Exercise 4 — gradient descent

Now glue it together. Each iteration:

1. Compute the current gradients from the current `(w, b)`.
2. Update both parameters: `w := w - alpha * dw` and `b := b - alpha * db`.
3. **Critical:** compute *both* gradients before doing *either* update. If you update `w` first and then compute `b`'s gradient using the new `w`, you've changed the algorithm into something wrong. Compute first, then update.
4. Record the new cost in `history`.

Run for 1000 iterations starting from `(w=0, b=0)` with `alpha=0.05`. The expected outcome: `w` settles near 2 and `b` near 5, recovering the true line we sampled from.


In [ ]:
def gradient_descent(X, y, w_init, b_init, alpha, n_iters):
    """Run gradient descent for n_iters steps.

    At each step, update both w and b *simultaneously* using the gradients
    computed from the CURRENT (w, b). This is the part that bites people.

    Returns:
        (w, b, history) where history is a list of cost values, one per iteration.
    """
    w, b = w_init, b_init
    history = []
    for i in range(n_iters):
        # TODO: compute gradients, then update w and b simultaneously.
        # Then append the new cost to history.
        pass
    return w, b, history


w_final, b_final, hist = gradient_descent(X, y, w_init=0.0, b_init=0.0, alpha=0.05, n_iters=1000)
print(f"After 1000 iterations with alpha=0.05:")
print(f"  w = {w_final:.4f}   (expected ≈ 1.99)")
print(f"  b = {b_final:.4f}   (expected ≈ 5.39)")
print(f"  final cost = {hist[-1] if hist else 'N/A'}   (expected ≈ 0.15)")


## Visualize the result

Two plots:

1. **Cost over iterations** — should drop fast at first, then level off. Log-scale on the y-axis makes the shape clearer.
2. **Final fit** — your green line should hug the data points. If the green line goes through the cloud, you've successfully fit a model.


In [ ]:
# Plot the cost trajectory and the final fitted line
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cost over iterations
axes[0].plot(hist, color="#4338ca")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("cost J(w, b)")
axes[0].set_title("Cost decreasing over iterations")
axes[0].set_yscale("log")
axes[0].grid(alpha=0.3)

# Data + fitted line
xs = np.linspace(0, 6, 100)
axes[1].scatter(X, y, color="#4338ca", s=80, zorder=3, label="training data")
axes[1].plot(xs, predict(xs, w_final, b_final), color="#10b981", lw=2, label=f"fit: y = {w_final:.2f}x + {b_final:.2f}")
axes[1].set_xlabel("X")
axes[1].set_ylabel("y")
axes[1].set_title("Final fit")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


## ⭐ Stretch goal — the learning rate

The learning rate `alpha` is the most important knob you'll ever tune. Too small and gradient descent crawls; too large and it overshoots and *diverges*.

**Try alphas of 0.001, 0.01, 0.1, 1.0, and 10.0.** For each, run 1000 iterations from `(0, 0)` and print the final `w`, `b`, and final cost. What do you see?

(Hint: at least one of those values will produce `nan` or astronomical numbers. That's divergence — the algorithm flying apart instead of settling down.)


In [ ]:
# STRETCH: try several learning rates and see what happens
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

# TODO: for each alpha, run gradient_descent for 1000 iterations starting at (0, 0).
# Print the final w, b, and final cost. Watch for divergence.
for alpha in alphas:
    pass  # replace with: w, b, hist = gradient_descent(...) ; print(...)


## Wrap-up

You just built linear regression from scratch in fewer than 30 lines of code. That's the whole game. Every fancier model — logistic regression, neural networks, the transformer behind ChatGPT — is structurally the same: a model, a cost, a gradient-based optimizer.

If you're ready, [next up is Week 2](course1-week2-multivariate.ipynb): regression with *multiple* input features, vectorization, and feature scaling.

The fully-worked solution is in [`solutions/course1-week1-gradient-descent-solution.ipynb`](../solutions/course1-week1-gradient-descent-solution.ipynb).
